# 한국수출입데이터_수집_V3
- V2를 기반으로 구조화/안정화한 업그레이드 버전
- HS 코드는 `EXPORT_CODE_ITEM` 딕셔너리에서 자동 로드
- (선택) DB에 **증분 업로드**: PK(date, root_hs_code, indicator)


In [1]:
import sys
from pathlib import Path
import os
import json
import time
import logging
import warnings
from dataclasses import dataclass
from typing import List, Optional, Tuple
from DATA.KEYS import KEYS

import requests
import xmltodict
import pandas as pd
import numpy as np
from pandas.tseries.offsets import MonthEnd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

# -----------------------------
# repo root 자동 탐색 + sys.path 등록
# -----------------------------
def add_repo_root_to_syspath(marker="DATA"):
    here = Path.cwd().resolve()
    for p in [here] + list(here.parents):
        if (p / marker).is_dir():
            if str(p) not in sys.path:
                sys.path.insert(0, str(p))
            return p
    raise RuntimeError("repo root(DATA 폴더)를 찾지 못했습니다.")

REPO_ROOT = add_repo_root_to_syspath("DATA")
print("REPO_ROOT =", REPO_ROOT)

# -----------------------------
# HS 코드-품목명 딕셔너리 import
# (export_top500_code_item_dict.py 를 DATA 폴더에 넣어둔 상태 기준)
# -----------------------------
from DATA.export_top500_code_item_dict import EXPORT_TOP500_CODE_ITEM
EXPORT_CODE_ITEM = EXPORT_TOP500_CODE_ITEM

print("HS codes loaded:", len(EXPORT_CODE_ITEM))

# -----------------------------
# Logging
# -----------------------------
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger("korea_trade_v3_nodb")


REPO_ROOT = C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy
HS codes loaded: 500


In [2]:
@dataclass
class TradeCollectorConfig:
    service_key: str
    start_year: int = 2024
    end_year: int = 2025
    max_workers: int = 12
    request_delay: float = 0.05
    retries: int = 3
    timeout: int = 30

    hs_include: Optional[List[str]] = None
    hs_exclude: Optional[List[str]] = None
    hs_limit: Optional[int] = None

    region_name: str = "전국"

SERVICE_KEY = KEYS["ODPE"]

# ✅ 서비스키는 환경변수 권장
# SERVICE_KEY = os.getenv("ODP_SERVICE_KEY", "YOUR_SERVICE_KEY_HERE")

cfg = TradeCollectorConfig(
    service_key=SERVICE_KEY,
    start_year=2007,
    end_year=2025,
    max_workers=12,
    request_delay=0.05,
    retries=3,
    timeout=30,
    hs_include=None,     # 예: ["854232","854231"]
    hs_exclude=None,     # 예: ["271019"]
    hs_limit=None,       # 예: 50
    region_name="전국",
)

HS_ALL = [str(k) for k in EXPORT_CODE_ITEM.keys()]

In [3]:
def build_hs_list(cfg: TradeCollectorConfig) -> List[str]:
    hs = HS_ALL.copy()

    if cfg.hs_include:
        inc = set(map(str, cfg.hs_include))
        hs = [h for h in hs if h in inc]

    if cfg.hs_exclude:
        exc = set(map(str, cfg.hs_exclude))
        hs = [h for h in hs if h not in exc]

    if cfg.hs_limit is not None:
        hs = hs[: int(cfg.hs_limit)]

    # sanitize
    hs = [str(h).strip().replace(".0", "") for h in hs if str(h).strip()]
    return hs


def yyyymm_range(start_year: int, end_year: int) -> List[str]:
    out = []
    for y in range(start_year, end_year + 1):
        for m in range(1, 13):
            out.append(f"{y}{m:02d}")
    return out


def split_yearly(period_list: List[str]) -> Tuple[List[str], List[str]]:
    # 1회 요청을 12개월(1년) 단위로 묶기
    start_list = [period_list[i] for i in range(0, len(period_list), 12)]
    end_list = [period_list[min(i + 11, len(period_list) - 1)] for i in range(0, len(period_list), 12)]
    return start_list, end_list


In [4]:
API_BASE = "https://apis.data.go.kr/1220000/Itemtrade/getItemtradeList"

def fetch_itemtrade(session: requests.Session, cfg: TradeCollectorConfig, start: str, end: str, hs_code: str) -> pd.DataFrame:
    url = (
        f"{API_BASE}"
        f"?serviceKey={cfg.service_key}"
        f"&strtYymm={start}"
        f"&endYymm={end}"
        f"&hsSgn={hs_code}"
    )

    for attempt in range(cfg.retries):
        try:
            resp = session.get(url, timeout=cfg.timeout)
            resp.raise_for_status()

            json_dict = json.loads(json.dumps(xmltodict.parse(resp.text), indent=2))
            items = json_dict.get("response", {}).get("body", {}).get("items")

            if not items or items.get("item") is None:
                return pd.DataFrame()

            item_data = items["item"]
            if isinstance(item_data, dict):
                item_data = [item_data]

            df = pd.DataFrame(item_data)
            df["root_hs_code"] = str(hs_code)
            df["item_name"] = EXPORT_CODE_ITEM.get(str(hs_code))

            if cfg.request_delay > 0:
                time.sleep(cfg.request_delay)

            return df

        except requests.RequestException as e:
            logger.warning(f"fetch fail {attempt+1}/{cfg.retries}: hs={hs_code}, {start}-{end}, err={e}")
            if attempt < cfg.retries - 1:
                time.sleep(2 ** attempt)
            else:
                return pd.DataFrame()
        except Exception as e:
            logger.error(f"unexpected: hs={hs_code}, {start}-{end}, err={e}")
            return pd.DataFrame()


In [5]:
def process_and_aggregate(raw: pd.DataFrame, region_name: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    원자료 -> 월/분기 집계 (expDlr, impDlr, balPayments 합)
    """
    if raw.empty:
        return pd.DataFrame(), pd.DataFrame()

    df = raw.copy()

    # 총계 제거(있을 때)
    if "year" in df.columns:
        df = df[df["year"] != "총계"].copy()

    # date 생성: year 컬럼('2024.01') 우선, 없으면 statYymm 사용
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        if "year" in df.columns:
            df["date"] = pd.to_datetime(df["year"].astype(str).str.replace(".", "-", regex=False), errors="coerce") + MonthEnd(0)
        elif "statYymm" in df.columns:
            df["date"] = pd.to_datetime(df["statYymm"].astype(str) + "01", format="%Y%m%d", errors="coerce") + MonthEnd(0)
        else:
            raise ValueError("date 생성 불가: year 또는 statYymm 컬럼이 필요합니다.")

    df = df.dropna(subset=["date"])
    df["new_year"] = df["date"].dt.year
    df["new_quarter"] = df["date"].dt.quarter
    df["new_month"] = df["date"].dt.month

    # 수치형 변환 (API에서 문자열로 올 수 있음)
    numeric_cols = ["balPayments", "expDlr", "impDlr"]
    for col in numeric_cols:
        if col not in df.columns:
            df[col] = 0.0
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0)

    df["root_hs_code"] = df["root_hs_code"].astype(str)

    agg_dict = {"balPayments": "sum", "expDlr": "sum", "impDlr": "sum"}

    # 월 집계
    m = (df.groupby(["root_hs_code", "item_name", "new_year", "new_quarter", "new_month"], dropna=False)
           .agg(agg_dict)
           .reset_index())
    m["region"] = region_name
    m["date"] = pd.to_datetime(m["new_year"].astype(str) + "-" + m["new_month"].astype(str) + "-01") + MonthEnd(0)

    # 분기 집계
    q = (df.groupby(["root_hs_code", "item_name", "new_year", "new_quarter"], dropna=False)
           .agg(agg_dict)
           .reset_index())
    q["region"] = region_name
    end_month_map = {1: "03", 2: "06", 3: "09", 4: "12"}
    q_end_month = q["new_quarter"].map(end_month_map)
    q["date"] = pd.to_datetime(q["new_year"].astype(str) + "-" + q_end_month + "-01") + MonthEnd(0)

    return q, m


def add_yoy_growth(df: pd.DataFrame, steps: int) -> pd.DataFrame:
    """
    YoY: 월(12), 분기(4)
    """
    if df.empty:
        return df

    out = df.copy().sort_values(["root_hs_code", "date"])
    g = out.groupby("root_hs_code", dropna=False)
    out["expDlr_yoy"] = g["expDlr"].transform(lambda x: x.pct_change(periods=steps))
    out["impDlr_yoy"] = g["impDlr"].transform(lambda x: x.pct_change(periods=steps))
    return out


def reshape_to_long(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df

    id_vars = ["date", "root_hs_code", "item_name", "region"]
    value_vars = ["balPayments", "expDlr", "impDlr", "expDlr_yoy", "impDlr_yoy"]
    value_vars = [c for c in value_vars if c in df.columns]

    long_df = df.melt(id_vars=id_vars, value_vars=value_vars, var_name="indicator", value_name="value")
    long_df = long_df.dropna(subset=["value"])
    long_df["root_hs_code"] = long_df["root_hs_code"].astype(str)
    return long_df


In [6]:
def run_trade_collector_no_db(cfg: TradeCollectorConfig):
    hs_codes = build_hs_list(cfg)
    if not hs_codes:
        raise ValueError("HS_CODES가 비어 있습니다. hs_include/hs_exclude/hs_limit 확인하세요.")

    periods = yyyymm_range(cfg.start_year, cfg.end_year)
    start_list, end_list = split_yearly(periods)

    logger.info(f"Period: {periods[0]} ~ {periods[-1]} (months={len(periods)})")
    logger.info(f"HS codes: {len(hs_codes)} | workers={cfg.max_workers}")

    session = requests.Session()
    all_dfs = []
    total_jobs = len(hs_codes) * len(start_list)

    with ThreadPoolExecutor(max_workers=cfg.max_workers) as ex:
        futures = {}
        for hs in hs_codes:
            for s, e in zip(start_list, end_list):
                fut = ex.submit(fetch_itemtrade, session, cfg, s, e, hs)
                futures[fut] = (hs, s, e)

        for fut in tqdm(as_completed(futures), total=len(futures), desc="API requests"):
            hs, s, e = futures[fut]
            try:
                df = fut.result(timeout=60)
                if df is not None and not df.empty:
                    all_dfs.append(df)
            except Exception as err:
                logger.warning(f"failed: hs={hs} {s}-{e} err={err}")

    session.close()

    if not all_dfs:
        logger.warning("수집된 데이터가 없습니다.")
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    raw = pd.concat(all_dfs, ignore_index=True)
    logger.info(f"Raw rows: {len(raw):,}")

    # 월/분기 집계
    q_df, m_df = process_and_aggregate(raw, cfg.region_name)

    # YoY
    m_yoy = add_yoy_growth(m_df, steps=12)
    q_yoy = add_yoy_growth(q_df, steps=4)

    # Long format
    m_long = reshape_to_long(m_yoy)
    q_long = reshape_to_long(q_yoy)

    return raw, m_yoy, q_yoy, m_long, q_long


t0 = time.time()
raw_df, monthly_yoy, quarterly_yoy, monthly_long, quarterly_long = run_trade_collector_no_db(cfg)
logger.info(f"Done. elapsed={time.time()-t0:.1f}s")

display(monthly_long.head(10))
display(quarterly_long.head(10))


2026-02-09 17:39:21,541 - INFO - Period: 200701 ~ 202512 (months=228)
2026-02-09 17:39:21,543 - INFO - HS codes: 500 | workers=12
API requests:   3%|▎         | 305/9500 [00:50<45:09,  3.39it/s]2026-02-09 17:40:11,978 - WARNING - Connection pool is full, discarding connection: apis.data.go.kr. Connection pool size: 10
2026-02-09 17:40:11,980 - WARNING - Connection pool is full, discarding connection: apis.data.go.kr. Connection pool size: 10
API requests:  15%|█▌        | 1464/9500 [03:49<38:17,  3.50it/s]2026-02-09 17:43:11,638 - WARNING - Connection pool is full, discarding connection: apis.data.go.kr. Connection pool size: 10
2026-02-09 17:43:11,640 - WARNING - Connection pool is full, discarding connection: apis.data.go.kr. Connection pool size: 10
API requests:  21%|██        | 1978/9500 [05:06<37:56,  3.30it/s]2026-02-09 17:44:28,540 - WARNING - Connection pool is full, discarding connection: apis.data.go.kr. Connection pool size: 10
2026-02-09 17:44:28,545 - WARNING - Connection

,date,root_hs_code,item_name,region,indicator,value
0,2012-01-31,121221,식용,전국,balPayments,7470206.0
1,2012-02-29,121221,식용,전국,balPayments,7810897.0
2,2012-03-31,121221,식용,전국,balPayments,13769031.0
3,2012-04-30,121221,식용,전국,balPayments,14188018.0
4,2012-05-31,121221,식용,전국,balPayments,15146482.0
5,2012-06-30,121221,식용,전국,balPayments,15440226.0
6,2012-07-31,121221,식용,전국,balPayments,14975784.0
7,2012-08-31,121221,식용,전국,balPayments,10395879.0
8,2012-09-30,121221,식용,전국,balPayments,10484071.0
9,2012-10-31,121221,식용,전국,balPayments,8880744.0


,date,root_hs_code,item_name,region,indicator,value
0,2012-03-31,121221,식용,전국,balPayments,29050134.0
1,2012-06-30,121221,식용,전국,balPayments,44774726.0
2,2012-09-30,121221,식용,전국,balPayments,35855734.0
3,2012-12-31,121221,식용,전국,balPayments,25105809.0
4,2013-03-31,121221,식용,전국,balPayments,26988446.0
5,2013-06-30,121221,식용,전국,balPayments,39301360.0
6,2013-09-30,121221,식용,전국,balPayments,28952290.0
7,2013-12-31,121221,식용,전국,balPayments,24963628.0
8,2014-03-31,121221,식용,전국,balPayments,28172599.0
9,2014-06-30,121221,식용,전국,balPayments,27103603.0


In [8]:
monthly_long.tail(12)

,date,root_hs_code,item_name,region,indicator,value
526953,2025-01-31,970191,회화ㆍ데생ㆍ파스텔,전국,impDlr_yoy,-0.492041
526954,2025-02-28,970191,회화ㆍ데생ㆍ파스텔,전국,impDlr_yoy,-0.482552
526955,2025-03-31,970191,회화ㆍ데생ㆍ파스텔,전국,impDlr_yoy,0.455944
526956,2025-04-30,970191,회화ㆍ데생ㆍ파스텔,전국,impDlr_yoy,-0.603188
526957,2025-05-31,970191,회화ㆍ데생ㆍ파스텔,전국,impDlr_yoy,0.085196
526958,2025-06-30,970191,회화ㆍ데생ㆍ파스텔,전국,impDlr_yoy,-0.436099
526959,2025-07-31,970191,회화ㆍ데생ㆍ파스텔,전국,impDlr_yoy,-0.433115
526960,2025-08-31,970191,회화ㆍ데생ㆍ파스텔,전국,impDlr_yoy,-0.166806
526961,2025-09-30,970191,회화ㆍ데생ㆍ파스텔,전국,impDlr_yoy,0.664723
526962,2025-10-31,970191,회화ㆍ데생ㆍ파스텔,전국,impDlr_yoy,0.357300


#### 2. 수출 무게 수집

In [9]:
def process_and_aggregate_weight(raw: pd.DataFrame, region_name: str):
    """
    원자료 -> 월/분기 집계 (expWgt, impWgt, balPayments 합)
    * balPayments는 금액 기반일 가능성이 높지만, 원자료에 있으니 일단 같이 유지(원치 않으면 제거 가능)
    """
    if raw.empty:
        return pd.DataFrame(), pd.DataFrame()

    df = raw.copy()

    # 총계 제거(있을 때)
    if "year" in df.columns:
        df = df[df["year"] != "총계"].copy()

    # date 생성: year('2024.01') 우선, 없으면 statYymm 사용
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        if "year" in df.columns:
            df["date"] = pd.to_datetime(df["year"].astype(str).str.replace(".", "-", regex=False), errors="coerce") + MonthEnd(0)
        elif "statYymm" in df.columns:
            df["date"] = pd.to_datetime(df["statYymm"].astype(str) + "01", format="%Y%m%d", errors="coerce") + MonthEnd(0)
        else:
            raise ValueError("date 생성 불가: year 또는 statYymm 컬럼이 필요합니다.")

    df = df.dropna(subset=["date"])
    df["new_year"] = df["date"].dt.year
    df["new_quarter"] = df["date"].dt.quarter
    df["new_month"] = df["date"].dt.month

    # ✅ 중량 컬럼 수치형 변환
    numeric_cols = ["expWgt", "impWgt"]
    for col in numeric_cols:
        if col not in df.columns:
            df[col] = 0.0
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0)

    # (선택) balPayments가 있으면 같이 보관
    if "balPayments" not in df.columns:
        df["balPayments"] = 0.0
    df["balPayments"] = pd.to_numeric(df["balPayments"], errors="coerce").fillna(0.0)

    df["root_hs_code"] = df["root_hs_code"].astype(str)

    agg_dict = {"expWgt": "sum", "impWgt": "sum", "balPayments": "sum"}

    # 월 집계
    m = (df.groupby(["root_hs_code", "item_name", "new_year", "new_quarter", "new_month"], dropna=False)
           .agg(agg_dict)
           .reset_index())
    m["region"] = region_name
    m["date"] = pd.to_datetime(m["new_year"].astype(str) + "-" + m["new_month"].astype(str) + "-01") + MonthEnd(0)

    # 분기 집계
    q = (df.groupby(["root_hs_code", "item_name", "new_year", "new_quarter"], dropna=False)
           .agg(agg_dict)
           .reset_index())
    q["region"] = region_name
    end_month_map = {1: "03", 2: "06", 3: "09", 4: "12"}
    q_end_month = q["new_quarter"].map(end_month_map)
    q["date"] = pd.to_datetime(q["new_year"].astype(str) + "-" + q_end_month + "-01") + MonthEnd(0)

    return q, m


def add_yoy_growth_weight(df: pd.DataFrame, steps: int) -> pd.DataFrame:
    """
    YoY: 월(12), 분기(4)
    expWgt_yoy, impWgt_yoy 생성
    """
    if df.empty:
        return df

    out = df.copy().sort_values(["root_hs_code", "date"])
    g = out.groupby("root_hs_code", dropna=False)

    out["expWgt_yoy"] = g["expWgt"].transform(lambda x: x.pct_change(periods=steps))
    out["impWgt_yoy"] = g["impWgt"].transform(lambda x: x.pct_change(periods=steps))
    return out


def reshape_to_long_weight(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df

    id_vars = ["date", "root_hs_code", "item_name", "region"]
    value_vars = ["expWgt", "impWgt", "expWgt_yoy", "impWgt_yoy", "balPayments"]
    value_vars = [c for c in value_vars if c in df.columns]

    long_df = df.melt(id_vars=id_vars, value_vars=value_vars, var_name="indicator", value_name="value")
    long_df = long_df.dropna(subset=["value"])
    long_df["root_hs_code"] = long_df["root_hs_code"].astype(str)
    return long_df


def run_trade_collector_no_db_weight(cfg: TradeCollectorConfig):
    hs_codes = build_hs_list(cfg)
    if not hs_codes:
        raise ValueError("HS_CODES가 비어 있습니다. hs_include/hs_exclude/hs_limit 확인하세요.")

    periods = yyyymm_range(cfg.start_year, cfg.end_year)
    start_list, end_list = split_yearly(periods)

    logger.info(f"Period: {periods[0]} ~ {periods[-1]} (months={len(periods)})")
    logger.info(f"HS codes: {len(hs_codes)} | workers={cfg.max_workers}")

    session = requests.Session()
    all_dfs = []

    with ThreadPoolExecutor(max_workers=cfg.max_workers) as ex:
        futures = {}
        for hs in hs_codes:
            for s, e in zip(start_list, end_list):
                fut = ex.submit(fetch_itemtrade, session, cfg, s, e, hs)
                futures[fut] = (hs, s, e)

        for fut in tqdm(as_completed(futures), total=len(futures), desc="API requests"):
            hs, s, e = futures[fut]
            try:
                df = fut.result(timeout=60)
                if df is not None and not df.empty:
                    all_dfs.append(df)
            except Exception as err:
                logger.warning(f"failed: hs={hs} {s}-{e} err={err}")

    session.close()

    if not all_dfs:
        logger.warning("수집된 데이터가 없습니다.")
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    raw = pd.concat(all_dfs, ignore_index=True)
    logger.info(f"Raw rows: {len(raw):,}")

    # ✅ 중량 기준 월/분기 집계
    q_df, m_df = process_and_aggregate_weight(raw, cfg.region_name)

    # ✅ 중량 기준 YoY
    m_yoy = add_yoy_growth_weight(m_df, steps=12)
    q_yoy = add_yoy_growth_weight(q_df, steps=4)

    # ✅ 중량 기준 Long format
    m_long = reshape_to_long_weight(m_yoy)
    q_long = reshape_to_long_weight(q_yoy)

    return raw, m_yoy, q_yoy, m_long, q_long


In [10]:
t0 = time.time()
raw_df, monthly_yoy_w, quarterly_yoy_w, monthly_long_w, quarterly_long_w = run_trade_collector_no_db_weight(cfg)
logger.info(f"Done. elapsed={time.time()-t0:.1f}s")

display(monthly_long_w.head(10))
display(quarterly_long_w.head(10))

2026-02-09 18:05:32,844 - INFO - Period: 200701 ~ 202512 (months=228)
2026-02-09 18:05:32,845 - INFO - HS codes: 500 | workers=12
API requests:   2%|▏         | 159/9500 [00:29<41:59,  3.71it/s]2026-02-09 18:06:02,548 - WARNING - Connection pool is full, discarding connection: apis.data.go.kr. Connection pool size: 10
2026-02-09 18:06:02,551 - WARNING - Connection pool is full, discarding connection: apis.data.go.kr. Connection pool size: 10
API requests:  42%|████▏     | 3972/9500 [08:40<17:03,  5.40it/s]2026-02-09 18:14:13,499 - WARNING - Connection pool is full, discarding connection: apis.data.go.kr. Connection pool size: 10
2026-02-09 18:14:13,510 - WARNING - Connection pool is full, discarding connection: apis.data.go.kr. Connection pool size: 9
API requests:  55%|█████▌    | 5235/9500 [11:32<15:53,  4.47it/s]2026-02-09 18:17:05,831 - WARNING - Connection pool is full, discarding connection: apis.data.go.kr. Connection pool size: 10
2026-02-09 18:17:05,836 - WARNING - Connection 

,date,root_hs_code,item_name,region,indicator,value
0,2012-01-31,121221,식용,전국,expWgt,841468.0
1,2012-02-29,121221,식용,전국,expWgt,2709297.0
2,2012-03-31,121221,식용,전국,expWgt,8927127.0
3,2012-04-30,121221,식용,전국,expWgt,3112525.0
4,2012-05-31,121221,식용,전국,expWgt,2093912.0
5,2012-06-30,121221,식용,전국,expWgt,1678497.0
6,2012-07-31,121221,식용,전국,expWgt,1720148.0
7,2012-08-31,121221,식용,전국,expWgt,1352414.0
8,2012-09-30,121221,식용,전국,expWgt,1219233.0
9,2012-10-31,121221,식용,전국,expWgt,1375632.0


,date,root_hs_code,item_name,region,indicator,value
0,2012-03-31,121221,식용,전국,expWgt,12477892.0
1,2012-06-30,121221,식용,전국,expWgt,6884934.0
2,2012-09-30,121221,식용,전국,expWgt,4291795.0
3,2012-12-31,121221,식용,전국,expWgt,3538517.0
4,2013-03-31,121221,식용,전국,expWgt,11656292.0
5,2013-06-30,121221,식용,전국,expWgt,7390930.0
6,2013-09-30,121221,식용,전국,expWgt,3308653.0
7,2013-12-31,121221,식용,전국,expWgt,3967047.0
8,2014-03-31,121221,식용,전국,expWgt,11695763.0
9,2014-06-30,121221,식용,전국,expWgt,4345228.0


#### 3) 월별 HS코드별 kg당 수출단가 계산 (달러/kg)

In [11]:
def merge_value_and_weight(
    monthly_value_df: pd.DataFrame,
    monthly_weight_df: pd.DataFrame,
    on_cols=("date", "root_hs_code"),
) -> pd.DataFrame:
    """
    금액(expDlr, impDlr) DF와
    중량(expWgt, impWgt) DF를
    월·HS코드 기준으로 병합
    """
    # 필요한 컬럼만 선택
    w = monthly_weight_df[[
        "date", "root_hs_code", "expWgt", "impWgt"
    ]].copy()

    merged = (
        monthly_value_df
        .merge(w, on=list(on_cols), how="left")
    )

    return merged

def add_unit_prices(
    df: pd.DataFrame,
    min_weight: float = 0.0,
) -> pd.DataFrame:
    """
    수출단가(expDlr/expWgt), 수입단가(impDlr/impWgt) 계산
    """
    out = df.copy()

    # 숫자형 보정
    for col in ["expDlr", "impDlr", "expWgt", "impWgt"]:
        if col not in out.columns:
            out[col] = np.nan
        out[col] = pd.to_numeric(out[col], errors="coerce")

    # 분모(중량) 필터링: 0 또는 너무 작은 값 제거
    exp_denom = out["expWgt"].where(out["expWgt"] > min_weight, np.nan)
    imp_denom = out["impWgt"].where(out["impWgt"] > min_weight, np.nan)

    # 단가 계산
    out["exp_unit_price_usd_per_kg"] = out["expDlr"] / exp_denom
    out["imp_unit_price_usd_per_kg"] = out["impDlr"] / imp_denom

    # inf 방지
    out.replace([np.inf, -np.inf], np.nan, inplace=True)

    return out

def add_unit_price_yoy(
    df: pd.DataFrame,
    group_col: str = "root_hs_code",
    date_col: str = "date",
    exp_col: str = "exp_unit_price_usd_per_kg",
    imp_col: str = "imp_unit_price_usd_per_kg",
    periods: int = 12,
) -> pd.DataFrame:
    """
    HS코드별 월 단가 YoY 계산:
    - exp_unit_price_usd_per_kg_yoy = pct_change(12)
    - imp_unit_price_usd_per_kg_yoy = pct_change(12)
    """
    out = df.copy()

    # 타입 보정
    out[date_col] = pd.to_datetime(out[date_col], errors="coerce")
    out[exp_col] = pd.to_numeric(out.get(exp_col, np.nan), errors="coerce")
    out[imp_col] = pd.to_numeric(out.get(imp_col, np.nan), errors="coerce")

    # 정렬 후 그룹별 YoY
    out = out.sort_values([group_col, date_col])
    g = out.groupby(group_col, dropna=False)

    out[f"{exp_col}_yoy"] = g[exp_col].transform(lambda s: s.pct_change(periods=periods))
    out[f"{imp_col}_yoy"] = g[imp_col].transform(lambda s: s.pct_change(periods=periods))

    # inf 방지
    out.replace([np.inf, -np.inf], np.nan, inplace=True)

    return out


In [12]:
monthly_merged = merge_value_and_weight(monthly_yoy, monthly_yoy_w)
monthly_with_unit_price = add_unit_prices(monthly_merged)
monthly_with_unit_price_yoy = add_unit_price_yoy(monthly_with_unit_price)


C:\Users\82108\AppData\Local\Temp\ipykernel_24356\3205092406.py:75: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  out[f"{exp_col}_yoy"] = g[exp_col].transform(lambda s: s.pct_change(periods=periods))
C:\Users\82108\AppData\Local\Temp\ipykernel_24356\3205092406.py:76: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  out[f"{imp_col}_yoy"] = g[imp_col].transform(lambda s: s.pct_change(periods=periods))


In [13]:
def to_long_monthly_trade(df: pd.DataFrame) -> pd.DataFrame:
    """
    monthly_with_unit_price_yoy (wide) -> long format
    기본 출력: date, root_hs_code, indicator, value
    (추가로 item_name, region이 있으면 같이 유지)
    """
    out = df.copy()

    # 필수 컬럼 체크/정리
    if "date" not in out.columns or "root_hs_code" not in out.columns:
        raise ValueError("df에는 최소한 'date', 'root_hs_code' 컬럼이 있어야 합니다.")

    out["date"] = pd.to_datetime(out["date"], errors="coerce")
    out["root_hs_code"] = out["root_hs_code"].astype(str)

    # id_vars: 가능한 한 유지 (DB에 item_name/region 저장하려면 같이 넣어도 됨)
    id_vars = ["date", "root_hs_code"]
    for c in ["item_name", "region"]:
        if c in out.columns:
            id_vars.append(c)

    # value_vars: 존재하는 지표만 자동 수집
    candidate_vars = [
        "balPayments",
        "expDlr", "impDlr",
        "expDlr_yoy", "impDlr_yoy",
        "expWgt", "impWgt",
        "expWgt_yoy", "impWgt_yoy",
        "exp_unit_price_usd_per_kg",
        "imp_unit_price_usd_per_kg",
        "exp_unit_price_usd_per_kg_yoy",
        "imp_unit_price_usd_per_kg_yoy",
    ]
    value_vars = [c for c in candidate_vars if c in out.columns]

    if not value_vars:
        raise ValueError("long으로 변환할 지표(value_vars)가 없습니다. 컬럼명을 확인하세요.")

    # melt -> long
    long_df = out.melt(
        id_vars=id_vars,
        value_vars=value_vars,
        var_name="indicator",
        value_name="value"
    )

    # 정리: inf/NaN 제거(원하면 NaN 유지 가능)
    long_df["value"] = pd.to_numeric(long_df["value"], errors="coerce")
    long_df.replace([np.inf, -np.inf], np.nan, inplace=True)

    # DB 저장 목적이면 보통 NULL은 빼는 게 좋음
    long_df = long_df.dropna(subset=["value"])

    # 컬럼 순서: date, root_hs_code, indicator, value (요청하신 구조 우선)
    base_cols = ["date", "root_hs_code", "indicator", "value"]
    extra_cols = [c for c in ["item_name", "region"] if c in long_df.columns]
    long_df = long_df[base_cols + extra_cols]

    # 정렬(가독성)
    long_df = long_df.sort_values(["date", "root_hs_code", "indicator"]).reset_index(drop=True)

    return long_df

In [14]:
monthly_long_all = to_long_monthly_trade(monthly_with_unit_price_yoy)
monthly_long_all.head(20)

,date,root_hs_code,indicator,value,item_name,region
0,2007-01-31,151800,balPayments,-2.010220e+05,동물성ㆍ식물성ㆍ미생물성 지방과 기름 및 이들의 분획물(끓이거나 산화ㆍ탈수ㆍ황화ㆍ취입...,전국
1,2007-01-31,151800,expDlr,7.613540e+05,동물성ㆍ식물성ㆍ미생물성 지방과 기름 및 이들의 분획물(끓이거나 산화ㆍ탈수ㆍ황화ㆍ취입...,전국
2,2007-01-31,151800,expWgt,6.455640e+05,동물성ㆍ식물성ㆍ미생물성 지방과 기름 및 이들의 분획물(끓이거나 산화ㆍ탈수ㆍ황화ㆍ취입...,전국
3,2007-01-31,151800,exp_unit_price_usd_per_kg,1.179363e+00,동물성ㆍ식물성ㆍ미생물성 지방과 기름 및 이들의 분획물(끓이거나 산화ㆍ탈수ㆍ황화ㆍ취입...,전국
4,2007-01-31,151800,impDlr,9.623760e+05,동물성ㆍ식물성ㆍ미생물성 지방과 기름 및 이들의 분획물(끓이거나 산화ㆍ탈수ㆍ황화ㆍ취입...,전국
5,2007-01-31,151800,impWgt,1.925397e+06,동물성ㆍ식물성ㆍ미생물성 지방과 기름 및 이들의 분획물(끓이거나 산화ㆍ탈수ㆍ황화ㆍ취입...,전국
6,2007-01-31,151800,imp_unit_price_usd_per_kg,4.998325e-01,동물성ㆍ식물성ㆍ미생물성 지방과 기름 및 이들의 분획물(끓이거나 산화ㆍ탈수ㆍ황화ㆍ취입...,전국
7,2007-01-31,190190,balPayments,-1.692640e+06,기타,전국
8,2007-01-31,190190,expDlr,7.085030e+05,기타,전국
9,2007-01-31,190190,expWgt,4.659690e+05,기타,전국


In [16]:
from sqlalchemy import text

from DATA.config import get_db_info, get_engine  # ✅ 여기서 host 포함 DB정보 가져옴

# ===============================
# 1) 테이블 생성 (없으면)
# ===============================
def ensure_table_long_v2(engine, table_name="korea_monthly_trade_data_v2"):
    ddl = f"""
    CREATE TABLE IF NOT EXISTS {table_name} (
        date DATE NOT NULL,
        root_hs_code VARCHAR(20) NOT NULL,
        indicator VARCHAR(80) NOT NULL,
        value DOUBLE NULL,
        item_name VARCHAR(255) NULL,
        region VARCHAR(50) NOT NULL DEFAULT '전국',
        created_at TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
        PRIMARY KEY (date, root_hs_code, indicator, region),
        KEY idx_indicator (indicator),
        KEY idx_hs (root_hs_code),
        KEY idx_date (date)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
    """
    with engine.begin() as conn:
        conn.execute(text(ddl))

# ===============================
# 2) DF 정규화
# ===============================
def normalize_monthly_long_all(df: pd.DataFrame) -> pd.DataFrame:
    need = ["date", "root_hs_code", "indicator", "value"]
    missing = [c for c in need if c not in df.columns]
    if missing:
        raise ValueError(f"DF에 필수 컬럼이 없습니다: {missing}")

    out = df.copy()

    out["date"] = pd.to_datetime(out["date"]).dt.date
    out["root_hs_code"] = out["root_hs_code"].astype(str)
    out["indicator"] = out["indicator"].astype(str)
    out["value"] = pd.to_numeric(out["value"], errors="coerce")

    if "item_name" not in out.columns:
        out["item_name"] = None
    if "region" not in out.columns:
        out["region"] = "전국"

    out["region"] = out["region"].fillna("전국").astype(str)

    # DF 내부 중복은 마지막 값만 남김(어차피 DB는 스킵 정책이지만 정리)
    out = out.sort_values(["date", "root_hs_code", "indicator", "region"])
    out = out.drop_duplicates(
        subset=["date", "root_hs_code", "indicator", "region"],
        keep="last"
    )

    out = out[["date", "root_hs_code", "indicator", "value", "item_name", "region"]]
    out = out.replace({np.nan: None})
    return out

# ===============================
# 3) INSERT IGNORE (중복이면 무조건 스킵)
# ===============================
def insert_ignore_long_df(
    df: pd.DataFrame,
    table_name="korea_monthly_trade_data_v2",
    chunksize=20000
):
    db_info = get_db_info()         # ✅ host 포함
    engine = get_engine(db_info)    # ✅ config.py의 엔진 생성 함수 사용

    df2 = normalize_monthly_long_all(df)

    cols = ["date", "root_hs_code", "indicator", "value", "item_name", "region"]
    sql = f"""
    INSERT IGNORE INTO {table_name} ({", ".join(cols)})
    VALUES ({", ".join(["%s"] * len(cols))})
    """

    inserted_total = 0
    skipped_est = 0

    with engine.begin() as conn:
        raw = conn.connection
        cur = raw.cursor()

        for start in range(0, len(df2), chunksize):
            chunk = df2.iloc[start:start + chunksize]
            data = [tuple(row[c] for c in cols) for _, row in chunk.iterrows()]

            cur.executemany(sql, data)
            inserted = cur.rowcount if cur.rowcount != -1 else 0

            inserted_total += inserted
            skipped_est += (len(chunk) - inserted)

        raw.commit()
        cur.close()

    print(f"[DB INSERT] total={len(df2):,} inserted≈{inserted_total:,} skipped≈{skipped_est:,}")

# ===============================
# 4) 사용 예시
# ===============================
# monthly_long_all = ... (현재 보유 DF)
# db_info = get_db_info()
# engine = get_engine(db_info)
# ensure_table_long_v2(engine)
# insert_ignore_long_df(monthly_long_all)


In [17]:
# ===============================
# 4) 사용 예시
# ===============================
# monthly_long_all = ... (현재 보유 DF)
db_info = get_db_info()
engine = get_engine(db_info)
ensure_table_long_v2(engine)
insert_ignore_long_df(monthly_long_all)

[DB INSERT] total=1,131,966 inserted≈1,131,966 skipped≈0
